# 01 — End-to-end IDS validation demonstration

**M7U5 — Development of Plugins for BIM Platforms — Zigurat Global Institute of Technology**

This notebook walks through the full pipeline:

1. Load both source IFC models
2. Run the hand-authored IDS ruleset (`ids/mep_services_v1.ids`) against each
3. Generate JSON / HTML / BCF / CSV reports
4. (Optional, requires `ANTHROPIC_API_KEY` in `.env`) Use Claude to translate one EIR clause into a new IDS specification
5. (Optional) Re-validate including the new spec
6. (Optional) Annotate a handful of non-conformances with plain-English remediation advice

Cells that require the LLM check the env at runtime and skip cleanly if the key is absent.

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.validator import Validator
from src.reporter import write_all

load_dotenv(REPO / ".env")
LLM_AVAILABLE = bool(os.environ.get("ANTHROPIC_API_KEY", "").startswith("sk-ant"))
LLM_AVAILABLE = LLM_AVAILABLE and not os.environ.get("ANTHROPIC_API_KEY", "").startswith("sk-ant-REPLACE")

IDS_PATH = REPO / "ids" / "mep_services_v1.ids"
IFC_PRIMARY = REPO / "ifc" / "Ifc4_Revit_MEP.ifc"
IFC_SECONDARY = REPO / "ifc" / "BoilerGasRadiatorDomesticHotWater.ifc"
OUT = REPO / "results"
OUT.mkdir(exist_ok=True)

print(f"Repo: {REPO}")
print(f"LLM cells will run: {LLM_AVAILABLE}")

Repo: D:\Projects\m7u5-ids-mep-validator
LLM cells will run: False


## Step 1 — Run the validator on the primary IFC

The primary model is the Autodesk Revit MEP Advanced Sample exported to IFC4 (28 MB, 302k entities, 8.5k distribution ports).

In [2]:
primary_run = Validator(IFC_PRIMARY, IDS_PATH).run()
primary_paths = write_all(primary_run, OUT)

primary_summary = pd.DataFrame([
    {
        "Specification": s["name"],
        "Applicable": s["total_applicable"],
        "Pass": s["total_applicable_pass"],
        "Fail": s["total_applicable_fail"],
        "% Pass": (round(100 * s["total_applicable_pass"] / s["total_applicable"], 1)
                   if s["total_applicable"] else None),
    }
    for s in primary_run.report["specifications"]
])
primary_summary

,Specification,Applicable,Pass,Fail,% Pass
0,Pipe segment — diameter and material,491,0,491,0.0
1,Distribution port — flow direction declared,8515,8515,0,100.0
2,Flow terminal — distribution system membership,11,0,11,0.0
3,Fire-suppression terminal — coverage area and ...,6,0,6,0.0
4,Pump — manufacturer and model identification,0,0,0,NaN
5,Duct segment — nominal size and material,837,0,837,0.0
6,Valve — operation type and predefined type,0,0,0,NaN
7,Cable segment — cross-sectional area declared,0,0,0,NaN
8,Electric appliance — nominal power declared,0,0,0,NaN


In [3]:
print(f"Open time:     {primary_run.open_seconds:.2f} s")
print(f"Validate time: {primary_run.validate_seconds:.2f} s")
print(f"Specs pass:    {primary_run.report['total_specifications_pass']}/{primary_run.report['total_specifications']}")
print(f"Checks pass:   {primary_run.report['percent_checks_pass']} %")

Open time:     1.84 s
Validate time: 0.36 s
Specs pass:    5/9
Checks pass:   76 %


## Step 2 — Inspect a handful of failures

Showing the first three failing entities under spec MEP-04 (fire-suppression terminals).

In [4]:
for spec in primary_run.report["specifications"]:
    if "fire-suppression" not in spec["name"].lower():
        continue
    print(f"Specification: {spec['name']}")
    for req in spec["requirements"]:
        for fail in req.get("failed_entities", [])[:3]:
            print(f"  - GlobalId={fail.get('global_id')}  class={fail.get('class')}")
            print(f"    Reason: {fail.get('reason')}")
    break

Specification: Fire-suppression terminal — coverage area and predefined type
  - GlobalId=1w91czu49BE8kLU1l3DYvH  class=IfcFireSuppressionTerminal
    Reason: The attribute value "NOTDEFINED" does not match the requirement
  - GlobalId=1w91czu49BE8kLU1l3DYvu  class=IfcFireSuppressionTerminal
    Reason: The attribute value "NOTDEFINED" does not match the requirement
  - GlobalId=1w91czu49BE8kLU1l3DYvc  class=IfcFireSuppressionTerminal
    Reason: The attribute value "NOTDEFINED" does not match the requirement
  - GlobalId=1w91czu49BE8kLU1l3DYvH  class=IfcFireSuppressionTerminal
    Reason: The required property set does not exist
  - GlobalId=1w91czu49BE8kLU1l3DYvu  class=IfcFireSuppressionTerminal
    Reason: The required property set does not exist
  - GlobalId=1w91czu49BE8kLU1l3DYvc  class=IfcFireSuppressionTerminal
    Reason: The required property set does not exist


## Step 3 — Run the validator on the secondary IFC

The secondary model is the EnEff:BIM Boiler / Gas Radiator / Domestic Hot Water VDI 6020 reference (1.35 MB, MIT-licensed). It exercises specs MEP-05 (pump) and MEP-07 (valve) which the primary model has zero instances of.

In [5]:
secondary_run = Validator(IFC_SECONDARY, IDS_PATH).run()
secondary_paths = write_all(secondary_run, OUT)

secondary_summary = pd.DataFrame([
    {
        "Specification": s["name"],
        "Applicable": s["total_applicable"],
        "Pass": s["total_applicable_pass"],
        "Fail": s["total_applicable_fail"],
        "% Pass": (round(100 * s["total_applicable_pass"] / s["total_applicable"], 1)
                   if s["total_applicable"] else None),
    }
    for s in secondary_run.report["specifications"]
])
secondary_summary

,Specification,Applicable,Pass,Fail,% Pass
0,Pipe segment — diameter and material,29,0,29,0.0
1,Distribution port — flow direction declared,114,114,0,100.0
2,Flow terminal — distribution system membership,0,0,0,NaN
3,Fire-suppression terminal — coverage area and ...,0,0,0,NaN
4,Pump — manufacturer and model identification,1,0,1,0.0
5,Duct segment — nominal size and material,0,0,0,NaN
6,Valve — operation type and predefined type,4,0,4,0.0
7,Cable segment — cross-sectional area declared,0,0,0,NaN
8,Electric appliance — nominal power declared,0,0,0,NaN


## Step 4 — LLM module: translate one EIR clause into a new IDS specification

Skipped automatically if `ANTHROPIC_API_KEY` is not set in `.env`.

In [6]:
if not LLM_AVAILABLE:
    print("Skipping: ANTHROPIC_API_KEY is not configured in .env.")
else:
    from src.llm_eir_to_ids import translate_clauses, write_output

    eir_clause = (
        "All air-handling units shall declare the supply-air design flow rate "
        "via Pset_AirHandlingUnitTypeCommon.NominalAirFlowRate so the "
        "commissioning team can verify against the room-by-room load schedule."
    )
    results = translate_clauses([eir_clause])
    for r in results:
        if r.error:
            print("Generation FAILED:", r.error)
        else:
            print("Generated <specification> XML:")
            print(r.spec_xml)
    generated_path = OUT / "generated_demo.ids"
    written = write_output(results, generated_path, title="LLM demo specification")
    print(f"\nWrote {written} specification(s) to {generated_path}")

Skipping: ANTHROPIC_API_KEY is not configured in .env.


## Step 5 — LLM module: annotate the first few failures with remediation

In [7]:
if not LLM_AVAILABLE:
    print("Skipping: ANTHROPIC_API_KEY is not configured in .env.")
else:
    from src.llm_remediation import annotate

    annotated = annotate(
        primary_paths.json,
        limit=5,
    )
    # Find the first failure that received a remediation note and show it.
    shown = 0
    for spec in annotated["specifications"]:
        for req in spec.get("requirements", []):
            for fail in req.get("failed_entities", []):
                advice = fail.get("remediation")
                if not advice:
                    continue
                print(f"Spec: {spec['name']}")
                print(f"Entity: {fail.get('class')} {fail.get('global_id')} - {fail.get('name')}")
                print(f"Reason: {fail.get('reason')}")
                print(f"Remediation: {advice}")
                print()
                shown += 1
                if shown >= 3:
                    break
            if shown >= 3:
                break
        if shown >= 3:
            break
    annotated_path = OUT / "Ifc4_Revit_MEP_remediated.json"
    annotated_path.write_text(json.dumps(annotated, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"Annotated report written to {annotated_path}")

Skipping: ANTHROPIC_API_KEY is not configured in .env.


## Step 6 — Combined summary

Side-by-side comparison of the two IFCs with pass rates per specification.

In [8]:
left = primary_summary.set_index("Specification").add_suffix(" (Revit MEP)")
right = secondary_summary.set_index("Specification").add_suffix(" (Boiler)")
combined = left.join(right)
combined

,Applicable (Revit MEP),Pass (Revit MEP),Fail (Revit MEP),% Pass (Revit MEP),Applicable (Boiler),Pass (Boiler),Fail (Boiler),% Pass (Boiler)
Specification,,,,,,,,
Pipe segment — diameter and material,491,0,491,0.0,29,0,29,0.0
Distribution port — flow direction declared,8515,8515,0,100.0,114,114,0,100.0
Flow terminal — distribution system membership,11,0,11,0.0,0,0,0,NaN
Fire-suppression terminal — coverage area and predefined type,6,0,6,0.0,0,0,0,NaN
Pump — manufacturer and model identification,0,0,0,NaN,1,0,1,0.0
Duct segment — nominal size and material,837,0,837,0.0,0,0,0,NaN
Valve — operation type and predefined type,0,0,0,NaN,4,0,4,0.0
Cable segment — cross-sectional area declared,0,0,0,NaN,0,0,0,NaN
Electric appliance — nominal power declared,0,0,0,NaN,0,0,0,NaN


## Step 7 — Reports written

Every file in `results/` is regenerated each time this notebook runs.

In [9]:
import os
rows = []
for path in sorted(OUT.glob("*")):
    rows.append({"file": path.name, "size_kb": round(path.stat().st_size / 1024, 1)})
pd.DataFrame(rows)

,file,size_kb
0,BoilerGasRadiatorDomesticHotWater_failures.csv,19.7
1,BoilerGasRadiatorDomesticHotWater_report.bcf,139.9
2,BoilerGasRadiatorDomesticHotWater_report.html,110.3
3,BoilerGasRadiatorDomesticHotWater_report.json,145.9
4,BoilerGasRadiatorDomesticHotWater_summary.csv,2.0
5,Ifc4_Revit_MEP_failures.csv,712.3
6,Ifc4_Revit_MEP_report.bcf,5347.4
7,Ifc4_Revit_MEP_report.html,127.9
8,Ifc4_Revit_MEP_report.json,7618.5
9,Ifc4_Revit_MEP_summary.csv,1.8
